# Vault Job Monitor
Monitor the status of a Veeva Vault job and send periodic updates to a Google Chat webhook.

### 1. Install Dependencies
Run the following cell to install `requests` and `python-dotenv`.

In [ ]:
!pip install requests python-dotenv

### 2. Import Libraries and Load Secrets
In the left sidebar of Google Colab, click the **Key icon (Secrets)** and add your environment variables (VAULT_DOMAIN, VEEVA_USERNAME, VEEVA_PASSWORD, CHAT_WEBHOOK_URL, etc.). Make sure to toggle "Notebook access" to ON for each secret.

In [ ]:
import requests
import time
import os
from google.colab import userdata

# Helper function to get environment variables or secrets
def get_config(key, default=None):
    try:
        return userdata.get(key)
    except userdata.SecretNotFoundError:
        return os.environ.get(key, default)

# --- Configuration ---
VAULT_DOMAIN = get_config("VAULT_DOMAIN", "https://cdms-vault-training.veevavault.com")
API_VERSION = get_config("API_VERSION", "v25.3")
JOB_ID = get_config("JOB_ID", "1122914")
CHAT_WEBHOOK_URL = get_config("CHAT_WEBHOOK_URL", "")
POLL_INTERVAL_SECONDS = int(get_config("POLL_INTERVAL_SECONDS", 600))
TOTAL_EXPECTED = int(get_config("TOTAL_EXPECTED", 268367))

# Credentials
USERNAME = get_config("VEEVA_USERNAME", "")
PASSWORD = get_config("VEEVA_PASSWORD", "")

# Global variable to hold the active session token
SESSION_ID = None

### 3. Define Monitor Functions

In [ ]:
def authenticate():
    """Authenticates with Veeva Vault and retrieves a new Session ID."""
    global SESSION_ID
    print("Authenticating with Veeva Vault...")
    
    url = f"{VAULT_DOMAIN}/api/{API_VERSION}/auth"
    payload = {
        "username": USERNAME,
        "password": PASSWORD
    }
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }
    
    response = requests.post(url, data=payload, headers=headers)
    response.raise_for_status() 
    
    data = response.json()
    if data.get("responseStatus") == "SUCCESS":
        SESSION_ID = data["sessionId"]
        print("Successfully authenticated.")
    else:
        raise Exception(f"Authentication failed. Check credentials: {data}")

def check_vault_job_status():
    """Checks the status of the job in Veeva Vault."""
    global SESSION_ID
    
    # If this is the first run and we don't have a session, authenticate
    if not SESSION_ID:
        authenticate()
        
    url = f"{VAULT_DOMAIN}/api/{API_VERSION}/services/jobs/{JOB_ID}"
    headers = {
        "Authorization": SESSION_ID,
        "Accept": "application/json"
    }
    
    response = requests.get(url, headers=headers)
    
    # Catch an expired session, re-authenticate, and retry the request
    if response.status_code == 401:
        print("Session expired or invalid. Attempting to re-authenticate...")
        authenticate()
        headers["Authorization"] = SESSION_ID
        response = requests.get(url, headers=headers)
        
    response.raise_for_status() # Catch any other HTTP errors
    
    data = response.json()
    
    if data.get("responseStatus") == "SUCCESS":
        return data["data"]["status"]
    else:
        raise Exception(f"Failed to fetch job: {data}")

def get_training_assignment_count():
    """Executes a VQL query to get the total number of training assignments created."""
    query = "SELECT id, status__v FROM training_assignment__v WHERE (training_requirement__v = 'V1G000000008001') PAGESIZE 0"
    url = f"{VAULT_DOMAIN}/api/{API_VERSION}/query"
    
    headers = {
        "Authorization": SESSION_ID,
        "Accept": "application/json"
    }
    params = {"q": query} 
    
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    
    data = response.json()
    if data.get("responseStatus") == "SUCCESS":
        return data["responseDetails"]["total"]
    else:
        print(f"⚠️ Warning: Failed to execute VQL: {data}")
        return "Error"

def send_chat_notification(status, total_created, is_finished=False):
    """Sends a formatted message to the Google Chat webhook."""
    if not CHAT_WEBHOOK_URL:
        print("⚠️ Warning: CHAT_WEBHOOK_URL not set. Skipping notification.")
        return

    if not is_finished:
        message_text = f"⏳ *Veeva Vault Polling Update*\nJob ID: `{JOB_ID}` is currently processing.\nCurrent Status: *{status}*\nTotal Created: *{total_created} / {TOTAL_EXPECTED}*"
    else:
        if status == "SUCCESS":
            message_text = f"✅ *Veeva Vault Update*\nJob ID: `{JOB_ID}` has finished running.\nFinal Status: *{status}*\nTotal Created: *{total_created} / {TOTAL_EXPECTED}*"
        else:
            message_text = f"⚠️ *Veeva Vault Alert*\nJob ID: `{JOB_ID}` ended with status: *{status}*.\nTotal Created: *{total_created} / {TOTAL_EXPECTED}*\nYou might want to check the logs."

    payload = {"text": message_text}
    response = requests.post(CHAT_WEBHOOK_URL, json=payload)
    
    if response.status_code == 429:
        print("⚠️ Warning: Google Chat is rate-limiting your webhook. Consider increasing your poll interval.")

### 4. Run the Monitor

In [ ]:
def monitor_job():
    """Main polling loop."""
    print(f"Starting monitor for Vault Job {JOB_ID}...")
    
    finished_statuses = ["SUCCESS", "ERRORS", "ABORTED", "CANCELLED"]
    
    while True:
        try:
            status = check_vault_job_status()
            print(f"[{time.strftime('%X')}] Current status: {status}")
            
            is_finished = status in finished_statuses
            
            # Get the current total created and send a chat message on EVERY check
            total_created = get_training_assignment_count()
            send_chat_notification(status, total_created, is_finished)
            
            if is_finished:
                print("Job finished. Final notification sent. Exiting.")
                break
                
            time.sleep(POLL_INTERVAL_SECONDS)
            
        except Exception as e:
            print(f"Monitor stopped due to an error: {e}")
            break

monitor_job()